# VULCAN Emulator — Extras Notebook

Self-contained demos and diagnostics for a trained FastChem emulator bundle.
Sections below are independent; set `MODEL` once, then run any section.

1. **Standalone inference** — simplest inference path (no `src/` imports).
2. **Emulator demo** — run on one saved test profile, compare with the stored target, save a plot.
3. **Saved test P-T profiles** — visualize the test-set T-P grid by source category.
4. **FastChem rerun comparison** — rerun FastChem and overlay on the stored profile.
5. **Chemistry diagnostic** — mean-absolute log10 error across transformer / stored / FastChem.

## Setup

In [ ]:
from __future__ import annotations

import subprocess
import sys
import tempfile
import types
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import h5py
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from matplotlib.lines import Line2D

# Allow running from either extras/ or the project root.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "extras":
    PROJECT_ROOT = PROJECT_ROOT.parent

# ----- The single knob for this notebook -----
MODEL = "fastchem_analytic_500k"
EXOGIBBS_ELEMENT_MODE = "fastchem_proxy"  # or "aas_fixed"

BUNDLE_PATH = PROJECT_ROOT / "models" / MODEL / "best_exported.npz"

# The bundle is weights + metadata only; the forward pass lives in
# src/models/transformer.py. ``src/`` ships alongside ``models/`` at the
# distribution root (``<dist>/src`` and ``<dist>/models/<run>/`` as
# siblings), so ``BUNDLE_PATH.parents[2]`` resolves to ``<dist>`` — add it
# to sys.path to pull in the inference code that shipped with these
# weights.
DIST_ROOT = BUNDLE_PATH.resolve().parents[2]
if not (DIST_ROOT / "src").is_dir():
    raise FileNotFoundError(
        f"No src/ found at {DIST_ROOT / 'src'}. The distribution must keep "
        "`src/` as a sibling of `models/` so the bundle and inference code "
        "ship together."
    )
sys.path.insert(0, str(DIST_ROOT))

from src.data_generation.data_loader import ProcessedSplit, load_processed_dataset
from src.data_generation.generation import (
    _copy_fastchem_runtime,
    list_run_ids_from_consolidated,
)
from src.data_generation.preprocess import inverse_block, inverse_mixed_block
from src.models.standalone_inference import load_model
from src.utils.helpers import resolve_path

PLOTS_DIR = BUNDLE_PATH.parent / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
STYLE_PATH = PROJECT_ROOT / "extras" / "science.mplstyle"
if STYLE_PATH.exists():
    plt.style.use(str(STYLE_PATH))

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"MODEL        : {MODEL}")
print(f"BUNDLE_PATH  : {BUNDLE_PATH}")
print(f"src (shipped): {DIST_ROOT / 'src'}")


## Shared utilities

Test-split / raw-dataset loading, FastChem rerun support, plot helpers.

In [ ]:
from src.models.classical_reference import (
    EPSILON,
    SOLAR_ELEMENT_ABUNDANCES,
    FastChemTestCase,
    FastChemTestContext,
    RawEquilibriumProfile,
    build_exogibbs_element_vector,
    build_exogibbs_species_indices,
    classify_temperature_profile_bucket,
    load_fastchem_raw_metadata_map,
    load_fastchem_test_case,
    load_fastchem_test_context,
    load_raw_equilibrium_profile,
    mean_abs_log10_error,
    resolve_vulcan_source_root,
    run_fastchem_online,
)

# Load the bundle once; every section below reuses `model`.
if not BUNDLE_PATH.exists():
    raise FileNotFoundError(f"Bundle not found: {BUNDLE_PATH}")
model = load_model(BUNDLE_PATH)
print(f"Loaded {model.model_type} ({model.chemistry_type}) with {len(model.data_contract['output_species_order'])} species.")
print(f"ExoGibbs mode: {EXOGIBBS_ELEMENT_MODE}")


## 1. Standalone inference

Simplest inference path: load the bundle and predict from physical inputs. The bundle holds only weights + metadata; the forward pass comes from the `src/` snapshot shipped alongside it (added to `sys.path` in the setup cell).

In [ ]:
standalone_model = load_model(BUNDLE_PATH)

# 50-level pressure grid matching the training domain: 100 -> 1e-7 bar.
pressure_bar = np.logspace(2, -7, 50)
temperature_k = np.full_like(pressure_bar, 1500.0)

predictions_log10 = np.asarray(
    standalone_model.predict_fastchem_profile(
        pressure_bar=pressure_bar,
        temperature_k=temperature_k,
        global_inputs=SOLAR_ELEMENT_ABUNDANCES,
        return_log10=True,
    )
)

idx = int(np.argmin(np.abs(pressure_bar - 0.1)))
print(f"Output shape: {predictions_log10.shape}")
print("Mixing ratios at P = 0.1 bar (log10):")
for name in ["H2", "H2O", "CO", "CO2", "CH4", "NH3", "H2S"]:
    if name in standalone_model.species:
        col = standalone_model.species.index(name)
        print(f"  {name:>5s} = {predictions_log10[idx, col]:+.3f}")

## 2. Emulator demo

Run the emulator on one saved processed test profile and plot it against the stored target.

In [ ]:
context = load_fastchem_test_context(BUNDLE_PATH, model.config, project_root=PROJECT_ROOT)
test_case = load_fastchem_test_case(context, str(np.random.default_rng().choice(context.split.run_ids)))
species = test_case.output_species


emulator_log10 = np.asarray(
    model.predict_fastchem_profile(
        pressure_bar=test_case.pressure_bar,
        temperature_k=test_case.temperature_k,
        global_inputs=test_case.global_inputs,
        return_log10=True,
    )
)
emulator_ymix = np.power(10.0, emulator_log10)
stored_log10 = np.log10(np.clip(test_case.stored_target_ymix, EPSILON, None))
phot_idx = int(np.argmin(np.abs(test_case.pressure_bar - 0.1)))

print(f"Run: {test_case.run_id}  ({test_case.pressure_bar.size} levels)")
print("Mixing ratios at P = 0.1 bar (log10):")
for name in ["H2", "H2O", "CO", "CO2", "CH4", "NH3", "H2S"]:
    if name in species:
        i = species.index(name)
        print(f"  {name:>5s}  stored = {stored_log10[phot_idx, i]:+.3f}, emulator = {emulator_log10[phot_idx, i]:+.3f}")

fig, (ax_pt, ax_mix) = plt.subplots(1, 2, figsize=(14, 7), sharey=True)
colors = plt.cm.tab20(np.linspace(0, 1, len(species)))

ax_pt.plot(test_case.temperature_k, test_case.pressure_bar, color="black", lw=2.0)
ax_pt.set_xlabel("Temperature [K]")
ax_pt.set_ylabel("Pressure [bar]")
ax_pt.set_yscale("log")
ax_pt.invert_yaxis()
ax_pt.set_xlim(0.0, 4000.0)
ax_pt.set_title("Stored Test P-T Profile")
ax_pt.yaxis.set_major_locator(mticker.LogLocator(base=10, numticks=8))
ax_pt.yaxis.set_minor_locator(mticker.NullLocator())

for i, name in enumerate(species):
    ax_mix.plot(np.clip(test_case.stored_target_ymix[:, i], EPSILON, None),
                test_case.pressure_bar, color=colors[i], lw=1.6, label=name)
    ax_mix.plot(np.clip(emulator_ymix[:, i], EPSILON, None),
                test_case.pressure_bar, color=colors[i], lw=1.2, ls="--")
ax_mix.set_xscale("log")
ax_mix.set_xlim(1.0e-20, 3.0)
ax_mix.set_xlabel("Mixing Ratio")
ax_mix.set_title("Stored Test vs Emulator")

species_legend = ax_mix.legend(fontsize=7, ncol=3, loc="lower left")
ax_mix.add_artist(species_legend)
ax_mix.legend(handles=[
    Line2D([0], [0], color="black", lw=1.6, label="Stored test"),
    Line2D([0], [0], color="black", lw=1.2, ls="--", label="Emulator"),
], fontsize=8, loc="upper right")
fig.suptitle(test_case.run_id, fontsize=11)
fig.tight_layout()

output_path = PLOTS_DIR / "emulator_demo.png"
fig.savefig(output_path, dpi=160)
plt.show()
print(f"Saved: {output_path}")

## 3. Saved test P-T profiles

Overlay test-set T-P profiles grouped by source (PT library vs analytic).

In [ ]:
PROFILES_PER_BUCKET = 5
BUCKET_STYLE = {
    "pt_library": {"cmap": "Reds", "ls": "-", "lw": 2.2, "label": "PT-library"},
    "analytic_radiative": {"cmap": "Blues", "ls": "--", "lw": 2.2, "label": "Analytic (radiative)"},
    "analytic_convective": {"cmap": "Purples", "ls": (0, (5, 2, 1, 2)), "lw": 2.5, "label": "Analytic (conv. adj.)"},
}

ctx_raw = load_fastchem_test_context(BUNDLE_PATH, model.config, project_root=PROJECT_ROOT, require_raw=True)
metadata_map = load_fastchem_raw_metadata_map(ctx_raw.raw_root, ctx_raw.split.run_ids)

buckets: dict[str, list[str]] = {n: [] for n in BUCKET_STYLE}
for rid in ctx_raw.split.run_ids:
    buckets[classify_temperature_profile_bucket(metadata_map[rid])].append(rid)

bucket_rng = np.random.default_rng()

fig, ax = plt.subplots(figsize=(8, 8))
for name, rids in buckets.items():
    style = BUCKET_STYLE[name]
    n_pick = min(PROFILES_PER_BUCKET, len(rids))
    picked = [str(r) for r in bucket_rng.choice(rids, size=n_pick, replace=False)] if n_pick else []
    cases = [load_fastchem_test_case(ctx_raw, rid) for rid in picked]
    cmap = plt.get_cmap(style["cmap"])
    palette = [cmap(0.35 + 0.55 * i / max(len(cases) - 1, 1)) for i in range(len(cases))]
    for i, case in enumerate(cases):
        ax.plot(case.temperature_k, case.pressure_bar,
                ls=style["ls"], lw=style["lw"], alpha=0.8, color=palette[i],
                label=style["label"] if i == 0 else None)

ax.set_yscale("log")
ax.set_ylim(1.0e2, 1.0e-7)
ax.set_xlim(0.0, 4000.0)
ax.set_xlabel("Temperature [K]")
ax.set_ylabel("Pressure [bar]")
ax.set_title("Saved Test P-T Profiles")
ax.legend(loc="best")

output_path = PLOTS_DIR / "saved_test_profiles.png"
fig.savefig(output_path)
plt.show()
print(f"Saved: {output_path}")

## 4. FastChem rerun comparison

Pick a raw test profile, rerun FastChem with identical inputs, and overlay.

In [ ]:
ctx_raw = load_fastchem_test_context(BUNDLE_PATH, model.config, project_root=PROJECT_ROOT, require_raw=True)
eligible_ids = [rid for rid in ctx_raw.split.run_ids if rid in ctx_raw.raw_run_ids]
run_id = str(np.random.default_rng().choice(eligible_ids))
profile = load_raw_equilibrium_profile(ctx_raw.raw_root, run_id)

fastchem_ymix = run_fastchem_online(
    source_root=resolve_vulcan_source_root(model.config, project_root=PROJECT_ROOT),
    pressure_bar=profile.pressure_bar,
    temperature_k=profile.temperature_k,
    globals_map=profile.globals,
    output_species=profile.output_species,
    config=model.config,
)

fig, (ax_pt, ax_mix, ax_delta) = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
colors = plt.cm.tab20(np.linspace(0, 1, len(profile.output_species)))
truth = np.clip(profile.equilibrium_ymix, EPSILON, None)
rerun = np.clip(fastchem_ymix, EPSILON, None)

ax_pt.plot(profile.temperature_k, profile.pressure_bar, color="black", lw=2.0)
ax_pt.set_xlabel("Temperature [K]")
ax_pt.set_ylabel("Pressure [bar]")
ax_pt.set_yscale("log")
ax_pt.invert_yaxis()
ax_pt.set_xlim(0.0, 3000.0)
ax_pt.set_title("Test P-T Profile")

for i, name in enumerate(profile.output_species):
    ax_mix.plot(truth[:, i], profile.pressure_bar, color=colors[i], lw=1.6, label=name)
    ax_mix.plot(rerun[:, i], profile.pressure_bar, color=colors[i], lw=1.2, ls="--")
    ax_delta.plot(np.log10(rerun[:, i] + EPSILON) - np.log10(truth[:, i] + EPSILON),
                  profile.pressure_bar, color=colors[i], lw=1.3)

ax_mix.set_xscale("log")
ax_mix.set_xlim(1.0e-20, 3.0)
ax_mix.set_xlabel("Mixing Ratio")
ax_mix.set_title("Mixing Ratios")
species_legend = ax_mix.legend(fontsize=7, ncol=3, loc="lower left")
ax_mix.add_artist(species_legend)
ax_mix.legend(handles=[
    Line2D([0], [0], color="black", lw=1.6, label="Test"),
    Line2D([0], [0], color="black", lw=1.2, ls="--", label="FastChem"),
], fontsize=8, loc="upper left")

ax_delta.axvline(0.0, color="black", lw=1.0, alpha=0.6)
ax_delta.set_xlim(-3, 3)
ax_delta.set_xlabel(r"$\log_{10}(\mathrm{FastChem}) - \log_{10}(\mathrm{Test})$")
ax_delta.set_title("FastChem Residual")

fig.suptitle(profile.run_id, fontsize=12)
fig.tight_layout()
output_path = PLOTS_DIR / f"{profile.run_id}_fastchem_compare.png"
fig.savefig(output_path, dpi=180)
plt.show()
print(f"Saved: {output_path}")

## 5. Three-way chemistry diagnostic

Compare the stored test target, live FastChem, ExoGibbs, and the exported
model on one random raw test profile. This is the notebook section that most
cleanly separates **bundle vs label** from **FastChem vs ExoGibbs**.


In [ ]:
from src.models.classical_reference import chemsetup_matched_to_fastchem
from exogibbs.api.equilibrium import (
    EquilibriumOptions,
    equilibrium_profile,
)
from exojax.utils.zsol import nsol

ctx_raw = load_fastchem_test_context(
    BUNDLE_PATH,
    model.config,
    project_root=PROJECT_ROOT,
    require_raw=True,
)
eligible_ids = [rid for rid in ctx_raw.split.run_ids if rid in ctx_raw.raw_run_ids]
run_id = str(np.random.default_rng().choice(eligible_ids))
test_case = load_fastchem_test_case(ctx_raw, run_id)
species = test_case.output_species

transformer_vmr = np.asarray(
    model.predict_fastchem_profile(
        pressure_bar=test_case.pressure_bar,
        temperature_k=test_case.temperature_k,
        global_inputs=test_case.global_inputs,
        return_log10=False,
    )
)
fastchem_vmr = run_fastchem_online(
    source_root=resolve_vulcan_source_root(model.config, project_root=PROJECT_ROOT),
    pressure_bar=test_case.pressure_bar,
    temperature_k=test_case.temperature_k,
    globals_map=test_case.raw_globals or test_case.global_inputs,
    output_species=species,
    config=model.config,
)
chem = chemsetup_matched_to_fastchem(resolve_vulcan_source_root(model.config, project_root=PROJECT_ROOT))
opts = EquilibriumOptions(epsilon_crit=1e-11, max_iter=1000, method="vmap_cold")
EG_IDX_17 = build_exogibbs_species_indices(chem, species)
solar_abundance = nsol()
exogibbs_vmr = np.asarray(
    equilibrium_profile(
        chem,
        test_case.temperature_k,
        test_case.pressure_bar,
        build_exogibbs_element_vector(
            chem,
            test_case.raw_globals or test_case.global_inputs,
            solar_abundances=solar_abundance,
            mode=EXOGIBBS_ELEMENT_MODE,
        ),
        Pref=1.0,
        options=opts,
    ).x[:, EG_IDX_17]
)
stored_vmr = test_case.stored_target_ymix
phot_idx = int(np.argmin(np.abs(test_case.pressure_bar - 0.1)))

print(f"Run: {test_case.run_id}  ({test_case.pressure_bar.size} levels)")

print("Mean |delta log10 VMR|:")
print(f"  transformer vs stored   = {mean_abs_log10_error(transformer_vmr, stored_vmr):.4f}")
print(f"  transformer vs FastChem = {mean_abs_log10_error(transformer_vmr, fastchem_vmr):.4f}")
print(f"  stored vs FastChem      = {mean_abs_log10_error(stored_vmr, fastchem_vmr):.4f}")
print(f"  FastChem vs ExoGibbs    = {mean_abs_log10_error(fastchem_vmr, exogibbs_vmr):.4f}")
print(f"  transformer vs ExoGibbs = {mean_abs_log10_error(transformer_vmr, exogibbs_vmr):.4f}")

print(f" VMR at P = 0.1 bar:")
print(f"  {'species':>6s}  {'transformer':>12s}  {'stored':>12s}  {'fastchem':>12s}  {'exogibbs':>12s}")
for name in ("H2", "He", "CO", "H2O", "CH4", "NH3", "H2S"):
    if name not in species:
        continue
    i = species.index(name)
    print(
        f"  {name:>6s}  {transformer_vmr[phot_idx, i]:12.4e}  "
        f"{stored_vmr[phot_idx, i]:12.4e}  {fastchem_vmr[phot_idx, i]:12.4e}  "
        f"{exogibbs_vmr[phot_idx, i]:12.4e}"
    )
